Diffusers and pipeline Code

In [1]:
from diffusers import StableDiffusion3Pipeline
import torch
from config import LAM_VALUES, TSR_DIR, PT_TSR_DIR, PROMPTS_FILE, MODEL_CACHE, SEED, TSR_SIGMA, SWAP_ALGORITHM, N_INF_STEPS, GUIDANCE_SCALE

    # ── Load model once ───────────────────────────────────────────────────────────
pipe = StableDiffusion3Pipeline.from_pretrained(
	"stabilityai/stable-diffusion-3-medium-diffusers",
	torch_dtype=torch.float16,
	cache_dir=MODEL_CACHE,
)
pipe = pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)

Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Imports, configs

In [2]:
from pathlib import Path
import gc
import pandas as pd
from tqdm import tqdm
import shutil

TSR_DIR     = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/images/new_tsr_samples_tester")
PT_TSR_DIR  = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/images/new_pt_samples_tester")

REPLICA_EXCHANGE = True
LAM_VALUES = [1.15, 1.1, 1.01]
INDEX_UNTIL = 10

gc.collect()
torch.cuda.empty_cache()
    

In [3]:
if PT_TSR_DIR.exists():
    shutil.rmtree(PT_TSR_DIR)


Sample!

In [4]:
# ── Load prompts once ─────────────────────────────────────────────────────────
prompts = pd.read_csv(PROMPTS_FILE, usecols=["text"], nrows=INDEX_UNTIL)["text"].tolist()
print(f"Loaded {len(prompts)} prompts")


replica_exchanges = [True, False]

lam_dirs = {}
for re in replica_exchanges:
	base = PT_TSR_DIR if re else TSR_DIR
	lam_dirs[re] = {l: base / f"lam{l:.3f}".replace(".", "p") for l in LAM_VALUES}
	for d in lam_dirs[re].values():
		d.mkdir(parents=True, exist_ok=True)

# ── Sweep ─────────────────────────────────────────────────────────────────────
for idx, prompt in enumerate(prompts):
	
	for replica_exchange in replica_exchanges:
	
		for tsr_lam in tqdm(LAM_VALUES, desc=f"idx={idx} re={replica_exchange}"):

			output_dir = lam_dirs[replica_exchange][tsr_lam]

			if (output_dir / f"{idx:05d}.png").exists():
				continue

			generator = torch.Generator(device="cuda").manual_seed(SEED)

			images = pipe(
				prompt,
				negative_prompt="",
				num_inference_steps=N_INF_STEPS,
				guidance_scale=GUIDANCE_SCALE,
				tsr_lam=tsr_lam,
				tsr_sigma=TSR_SIGMA,
				replica_exchange=replica_exchange,
				swap_algorithm=SWAP_ALGORITHM,
				generator=generator,
			).images

			out_path = output_dir / f"{idx:05d}.png"
			images[0].save(out_path, icc_profile=None)


			del images
		torch.cuda.empty_cache()

print("\n All k values complete.")

Loaded 10 prompts


idx=0 re=True:   0%|          | 0/3 [00:00<?, ?it/s]

 We tsr by 1.15 with replica exchange True
 f between 1.3798828125 and 0.724609375 mean -0.0838947668671608
 energy diff 0.2529194951057434 between lams s 0.724609375 and t 1.3798828125 temp s 1.122268557548523 t 0.8693490624427795
Time 787.79 swap btwn source 0.72 and target 1.38 accept 0.980 std 0.851
 f between 1.3798828125 and 0.724609375 mean -0.031749580055475235
 energy diff 0.2956632375717163 between lams s 0.724609375 and t 1.3798828125 temp s 1.1461091041564941 t 0.8504458665847778
Time 763.76 swap btwn source 0.72 and target 1.38 accept 0.991 std 0.848
 f between 1.3798828125 and 0.724609375 mean -0.18514299392700195
 energy diff 0.4652995467185974 between lams s 0.724609375 and t 1.3798828125 temp s 1.2493960857391357 t 0.7840965390205383
Time 648.86 swap btwn source 0.72 and target 1.38 accept 0.922 std 0.875
 f between 1.3798828125 and 0.724609375 mean -0.10529167950153351
 energy diff 0.5022703409194946 between lams s 0.724609375 and t 1.3798828125 temp s 1.2736501693725

idx=0 re=True:  33%|███▎      | 1/3 [00:26<00:52, 26.08s/it]

 We tsr by 1.10 with replica exchange True
 f between 1.3203125 and 0.7578125 mean -0.07203273475170135
 energy diff 0.2184322476387024 between lams s 0.7578125 and t 1.3203125 temp s 1.1059647798538208 t 0.8875325322151184
Time 787.79 swap btwn source 0.76 and target 1.32 accept 0.984 std 0.851
 f between 1.3203125 and 0.7578125 mean -0.026445098221302032
 energy diff 0.255399227142334 between lams s 0.7578125 and t 1.3203125 temp s 1.126268744468689 t 0.870869517326355
Time 763.76 swap btwn source 0.76 and target 1.32 accept 0.993 std 0.848
 f between 1.3203125 and 0.7578125 mean -0.15904662013053894
 energy diff 0.40135109424591064 between lams s 0.7578125 and t 1.3203125 temp s 1.212924599647522 t 0.8115735054016113
Time 648.86 swap btwn source 0.76 and target 1.32 accept 0.941 std 0.874
 f between 1.3203125 and 0.7578125 mean -0.08359553664922714
 energy diff 0.43290799856185913 between lams s 0.7578125 and t 1.3203125 temp s 1.2329704761505127 t 0.8000624775886536
Time 614.30 swa

...

 We tsr by 1.10 with replica exchange True
 f between 1.3203125 and 0.7578125 mean -0.044874683022499084
 energy diff 0.2184322476387024 between lams s 0.7578125 and t 1.3203125 temp s 1.1059647798538208 t 0.8875325322151184
Time 787.79 swap btwn source 0.76 and target 1.32 accept 0.990 std 0.806
 f between 1.3203125 and 0.7578125 mean -0.01566844992339611
 energy diff 0.255399227142334 between lams s 0.7578125 and t 1.3203125 temp s 1.126268744468689 t 0.870869517326355
Time 763.76 swap btwn source 0.76 and target 1.32 accept 0.996 std 0.793
 f between 1.3203125 and 0.7578125 mean -0.09664205461740494
 energy diff 0.40135109424591064 between lams s 0.7578125 and t 1.3203125 temp s 1.212924599647522 t 0.8115735054016113
Time 648.86 swap btwn source 0.76 and target 1.32 accept 0.963 std 0.756
 f between 1.3203125 and 0.7578125 mean -0.04738624766469002
 energy diff 0.43290799856185913 between lams s 0.7578125 and t 1.3203125 temp s 1.2329704761505127 t 0.8000624775886536
Time 614.30 swa

idx=4 re=True:  67%|██████▋   | 2/3 [00:51<00:25, 25.78s/it]

 We tsr by 1.01 with replica exchange True


idx=4 re=True:  67%|██████▋   | 2/3 [00:58<00:29, 29.24s/it]


KeyboardInterrupt: 

Fid computation

In [ ]:
from fid import compute_sweep
# from config import LAM_VALUES, TSR_DIR, PT_TSR_DIR, PROMPTS_FILE, MODEL_CACHE, SEED, TSR_SIGMA, SWAP_ALGORITHM, N_INF_STEPS, GUIDANCE_SCALE


compute_sweep(
	lam_values=LAM_VALUES,
	replica_exchanges=[True, False],
	device="cuda",
	target_indices=None,
	index_until = 20,
	pt_sr_dir = PT_TSR_DIR,
	tsr_dir = TSR_DIR,
)